In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("/kaggle/input/notebooks/jigishadas/ssi-creation/dataset_with_ssi.csv")

df.head()

,function_count,parameter_count,variable_count,declaration_count,if_count,for_count,while_count,switch_count,return_count,call_count,...,expression_density,variable_density,statement_density,nesting_factor,cyclomatic_estimate,avg_parameters,language,repository,function,SSI
0,1,2,2,1,1,0,0,0,0,4,...,0.025974,0.025974,0.025974,0.129870,2,2.0,java,molgenis/molgenis,L1Cache.put,-1.407771
1,1,0,7,4,1,1,0,0,1,1,...,0.010309,0.072165,0.010309,0.092784,3,0.0,java,belaban/JGroups,RequestTable.size,-0.685910
2,1,3,8,4,0,0,0,0,0,17,...,0.010601,0.028269,0.010601,0.045936,1,3.0,java,googleads/googleads-java-lib,DownloadCriteriaReportWithAwql.runExample,0.810129
3,1,4,0,0,0,0,0,0,1,2,...,0.000000,0.000000,0.000000,0.117647,1,4.0,java,Azure/azure-sdk-for-java,NetworkInterfacesInner.updateTagsAsync,-2.041741
4,2,6,4,2,6,0,0,0,3,12,...,0.002342,0.009368,0.002342,0.037471,7,3.0,java,Azure/azure-sdk-for-java,BlobContainersInner.extendImmutabilityPolicyWi...,2.781887


In [3]:
def evaluate_language(train_languages, test_language):

    train_df = df[df["language"].isin(train_languages)]
    test_df = df[df["language"] == test_language]

    X_train = train_df.drop(
        columns=[
            "language",
            "repository",
            "function",
            "SSI"
        ]
    )

    y_train = train_df["SSI"]

    X_test = test_df.drop(
        columns=[
            "language",
            "repository",
            "function",
            "SSI"
        ]
    )

    y_test = test_df["SSI"]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(mean_squared_error(y_test, pred))

    r2 = r2_score(y_test, pred)

    return mae, rmse, r2

In [4]:
results = []

languages = [
    "java",
    "python",
    "javascript",
    "go"
]

for test_lang in languages:

    train_langs = [
        l for l in languages
        if l != test_lang
    ]

    mae, rmse, r2 = evaluate_language(
        train_langs,
        test_lang
    )

    results.append({
        "Test Language": test_lang,
        "Train Languages": ", ".join(train_langs),
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

results_df = pd.DataFrame(results)

results_df

,Test Language,Train Languages,MAE,RMSE,R²
0,java,"python, javascript, go",0.197121,0.373065,0.983874
1,python,"java, javascript, go",0.256536,0.801184,0.942830
2,javascript,"java, python, go",0.298997,1.113083,0.934701
3,go,"java, python, javascript",0.244842,0.544258,0.966509
